In [ ]:
import pandas as pd

df = pd.read_csv("Data/climate_master.csv")

In [2]:
df['state'].unique()

array(['Telegana', 'West Bengal'], dtype=object)

In [3]:
import pandas as pd
import numpy as np
import calendar
from math import gamma, pi

# =============================================================
# STEP 1 — HARGREAVES PET (unchanged)
# =============================================================

def extraterrestrial_radiation(lat_deg, month):
    mid_days = [15, 46, 74, 105, 135, 166,
                196, 227, 258, 288, 319, 349]
    J     = mid_days[month - 1]
    phi   = lat_deg * pi / 180.0
    dr    = 1 + 0.033 * np.cos(2 * pi * J / 365)
    delta = 0.409 * np.sin(2 * pi * J / 365 - 1.39)
    arg     = np.clip(-np.tan(phi) * np.tan(delta), -1.0, 1.0)
    omega_s = np.arccos(arg)
    Ra_mj = (24 * 60 / pi) * 0.082 * dr * (
        omega_s * np.sin(phi) * np.sin(delta)
        + np.cos(phi) * np.cos(delta) * np.sin(omega_s)
    )
    return Ra_mj / 2.45


def hargreaves_pet(tmean, tmin, tmax, lat_deg, month, year):
    Ra_mm_day = extraterrestrial_radiation(lat_deg, month)
    n_days    = calendar.monthrange(year, month)[1]
    td        = np.maximum(tmax - tmin, 0.0)
    pet_daily = 0.0023 * Ra_mm_day * (tmean + 17.8) * np.sqrt(td)
    return pet_daily * n_days


# =============================================================
# STEP 2 — LOG-LOGISTIC FIT (FIXED)
# =============================================================

def fit_log_logistic_pwm(data):
    """
    Returns (alpha, beta, loc) or (None, None, None).

    FIX 1: beta must be > 1 for log-logistic to have a
            finite mean (physically required for SPEI).
    FIX 2: try/except around math.gamma since gamma(1-1/beta)
            raises ValueError when 1-1/beta is a non-positive
            integer (e.g. beta = 0.5 → gamma(-1) = undefined).
    """
    x = np.sort(data[~np.isnan(data)])
    n = len(x)

    if n < 4:
        return None, None, None

    i  = np.arange(1, n + 1)
    p  = (i - 0.35) / n

    w0 = np.mean(x)
    w1 = np.sum(p    * x) / n
    w2 = np.sum(p**2 * x) / n

    denom = 6*w1 - w0 - 6*w2
    if abs(denom) < 1e-12:
        return None, None, None

    beta = (2*w1 - w0) / denom

    # -------------------------------------------------------
    # FIX 1: beta must be strictly > 1
    # beta <= 0 → invalid shape
    # 0 < beta <= 1 → infinite mean; gamma(1-1/beta) undefined
    # -------------------------------------------------------
    if beta <= 1.0:
        return None, None, None

    # -------------------------------------------------------
    # FIX 2: guard against math.gamma edge-case crashes
    # -------------------------------------------------------
    try:
        g1 = gamma(1.0 + 1.0 / beta)
        g2 = gamma(1.0 - 1.0 / beta)
    except (ValueError, OverflowError):
        return None, None, None

    prod = g1 * g2
    if prod <= 0:
        return None, None, None

    alpha = (2*w1 - w0) * beta / prod
    loc   = w0 - alpha * prod

    if alpha <= 0:
        return None, None, None

    return alpha, beta, loc


# =============================================================
# STEP 3 — DISTRIBUTION HELPERS (unchanged)
# =============================================================

def log_logistic_cdf(x, alpha, beta, loc):
    z   = np.where((x - loc) > 0, (x - loc) / alpha, 1e-10)
    cdf = 1.0 / (1.0 + z**(-beta))
    return np.clip(cdf, 1e-10, 1 - 1e-10)


def normal_ppf(p):
    p   = np.asarray(p, dtype=float)
    c0, c1, c2 = 2.515517, 0.802853, 0.010328
    d1, d2, d3 = 1.432788, 0.189269, 0.001308

    def _approx(t):
        num = c0 + c1*t + c2*t**2
        den = 1.0 + d1*t + d2*t**2 + d3*t**3
        return t - num / den

    result  = np.zeros_like(p)
    mask_lo = (p <= 0.5)
    mask_hi = ~mask_lo
    t_lo    = np.sqrt(-2 * np.log(np.where(mask_lo, p,     1e-10)))
    t_hi    = np.sqrt(-2 * np.log(np.where(mask_hi, 1 - p, 1e-10)))
    result  = np.where(mask_lo, -_approx(t_lo), result)
    result  = np.where(mask_hi,  _approx(t_hi), result)
    return result


# =============================================================
# STEP 4 — EMPIRICAL FALLBACK (NEW)
# Used when log-logistic fit fails for a calendar month.
# Converts ranks → uniform plotting positions → normal scores.
# Preserves temporal order of SPEI values.
# =============================================================

def empirical_normal_scores(values):
    """
    For an array of values (may include NaN):
    - Rank the non-NaN entries
    - Map ranks to Blom plotting positions p = (r-0.375)/(n+0.25)
    - Return normal_ppf(p), NaN preserved
    """
    n      = len(values)
    result = np.full(n, np.nan)
    valid  = ~np.isnan(values)
    idx    = np.where(valid)[0]
    vals   = values[idx]

    # Rank (1-based), ties get average rank
    order  = np.argsort(vals)
    ranks  = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(vals) + 1)

    n_valid = len(vals)
    p       = (ranks - 0.375) / (n_valid + 0.25)
    result[idx] = normal_ppf(p)

    return result


# =============================================================
# STEP 5 — SPEI-3 CORE (FIXED)
# =============================================================

def compute_spei_3(D_series):
    """
    Compute SPEI-3.
    - Primary:  log-logistic via PWM
    - Fallback: empirical normal scores (when PWM fails)
    """
    D_accum  = D_series.rolling(window=3, min_periods=3).sum()
    spei_out = pd.Series(np.nan, index=D_series.index)

    for month in range(1, 13):

        idx    = D_accum.index[D_accum.index.month == month]
        values = D_accum.loc[idx].values

        if len(values[~np.isnan(values)]) < 4:
            continue

        alpha, beta, loc = fit_log_logistic_pwm(values)

        if alpha is not None:
            # --------------------------------------------------
            # Primary path: parametric log-logistic
            # Handle NaN inputs safely before passing to CDF
            # --------------------------------------------------
            safe   = np.where(np.isnan(values), loc + alpha, values)
            cdf    = log_logistic_cdf(safe, alpha, beta, loc)
            scores = normal_ppf(cdf)
            scores[np.isnan(values)] = np.nan

        else:
            # --------------------------------------------------
            # FIX 3 — Fallback path: empirical normal scores
            # Triggered for dry months where beta <= 1
            # --------------------------------------------------
            scores = empirical_normal_scores(values)

        spei_out.loc[idx] = scores

    return spei_out


# =============================================================
# STEP 6 — DROUGHT CLASSIFICATION (unchanged)
# =============================================================

def classify_spei(x):
    if pd.isna(x):      return np.nan
    elif x <= -1.5:     return "Severe Drought"
    elif x <= -1.0:     return "Moderate Drought"
    elif x <   1.0:     return "Normal"
    elif x <   1.5:     return "Moderately Wet"
    else:               return "Severely Wet"


# =============================================================
# STEP 7 — MAIN PIPELINE
# =============================================================

allowed_states = ["Telegana", "West Bengal"]
df = df[df["state"].isin(allowed_states)].copy()

df['date'] = pd.to_datetime(
    dict(year=df['year'], month=df['month'], day=1)
)

df = df.sort_values(
    ['state', 'district', 'date']
).reset_index(drop=True)

all_results = []

for (state, district), grp in df.groupby(['state', 'district']):

    grp = grp.copy().sort_values('date').set_index('date')
    lat = grp['Latitude'].iloc[0]

    try:
        grp['PET'] = [
            hargreaves_pet(
                tmean   = row['temp_mean'],
                tmin    = row['temp_min'],
                tmax    = row['temp_max'],
                lat_deg = lat,
                month   = row.name.month,
                year    = row.name.year
            )
            for _, row in grp.iterrows()
        ]
    except Exception as e:
        print(f"PET failed for {district} ({state}): {e}")
        continue

    grp['D'] = grp['rain_mean'] - grp['PET']

    D_clean = pd.Series(
        grp['D'].values,
        index=pd.date_range(
            start   = grp.index[0],
            periods = len(grp),
            freq    = 'MS'
        )
    )

    grp['SPEI_3'] = compute_spei_3(D_clean).values
    grp = grp.reset_index()
    all_results.append(grp)

# =============================================================
# STEP 8 — COMBINE, CLASSIFY, SAVE
# =============================================================

final_df = pd.concat(all_results, ignore_index=True)
final_df['SPEI3_Class'] = final_df['SPEI_3'].apply(classify_spei)

final_df.to_csv("NEW_district_spei3_output.csv", index=False)

print("Done.")
print(f"Shape : {final_df.shape}")

# Verify no months are fully NaN
print("\nNaN count by calendar month:")
print(
    final_df.groupby(final_df['date'].dt.month)['SPEI_3']
    .apply(lambda x: x.isna().sum())
)
print(final_df[['state','district','date','PET','D','SPEI_3','SPEI3_Class']].head(10))

Done.
Shape : (8624, 15)

NaN count by calendar month:
date
1     56
2     56
3      0
4      0
5      0
6      0
7      0
8      0
9      0
10     0
11     0
12     0
Name: SPEI_3, dtype: int64
      state  district       date        PET          D        SPEI_3  \
0  Telegana  Adilabad 2013-01-01  29.754980 -29.629983           NaN   
1  Telegana  Adilabad 2013-02-01  40.048356 -39.428012           NaN   
2  Telegana  Adilabad 2013-03-01  58.886138 -58.790427  1.161978e+00   
3  Telegana  Adilabad 2013-04-01  76.484021 -75.642676  1.899586e-01   
4  Telegana  Adilabad 2013-05-01  89.970175 -89.857463  1.010067e-07   
5  Telegana  Adilabad 2013-06-01  66.173300 -48.822970  1.673305e+00   
6  Telegana  Adilabad 2013-07-01  57.912971 -33.971342  1.673305e+00   
7  Telegana  Adilabad 2013-08-01  54.056239 -44.832828  1.673305e+00   
8  Telegana  Adilabad 2013-09-01  54.691337 -48.201172  1.673305e+00   
9  Telegana  Adilabad 2013-10-01  47.212980 -41.878328  8.482211e-01   

      SPEI3_